# Model Experiments

## Цель

На основе признаков, сформированных на этапе E03, сравнить несколько моделей машинного обучения и выбрать устойчивый вариант для финального решения задачи детекции ботов.

Основная метрика соревнования:

**Precision @ Recall >= 0.70**

Дополнительные диагностические метрики:

- PR-AUC;
- ROC-AUC.

## Принципы эксперимента

- используются только события внутри окна `[window_start_ts, window_end_ts)`;
- признаки доступны на момент окончания observation window;
- test не используется для выбора модели;
- основной validation split сохраняется по времени;
- сравниваются разные классы моделей:
  - Logistic Regression;
  - Random Forest;
  - HistGradientBoosting;
  - CatBoost;
- подбор гиперпараметров ограничен небольшим числом заранее заданных конфигураций;
- решение выбирается прежде всего по `Precision @ Recall >= 0.70`;
- после выбора проводится дополнительная временная проверка устойчивости;
- все эксперименты воспроизводимы при фиксированном `random_state`.

## Базовый feature set

В качестве основного набора используется E03:

- возраст cookie;
- интенсивность активности;
- разнообразие объектов;
- распределение типов событий;
- временные характеристики активности;
- интервалы между событиями;
- burstiness;
- активный временной диапазон.

In [101]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

from catboost import CatBoostClassifier

SEED = 42

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
EVENTS_PATH = DATA_DIR / "events.csv.gz"

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from metric import precision_at_recall

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

In [102]:
import platform
import sklearn
import catboost

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("CatBoost:", catboost.__version__)

Python: 3.12.6
NumPy: 2.1.1
Pandas: 2.2.3
scikit-learn: 1.7.2
CatBoost: 1.2.10


## Фиксация random state

Во всех моделях используется фиксированный `random_state`, чтобы результаты экспериментов можно было воспроизвести.

Основная метрика импортируется непосредственно из предоставленного `metric.py`, чтобы локальная оценка соответствовала официальной реализации.

In [103]:
print("SEED:", SEED)
print("ROOT_DIR:", ROOT_DIR)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("EVENTS_PATH:", EVENTS_PATH)

SEED: 42
ROOT_DIR: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge
TRAIN_PATH: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge\data\train.csv
TEST_PATH: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge\data\test.csv
EVENTS_PATH: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge\data\events.csv.gz


## Загрузка train, test и событий

Загружаются исходные таблицы.

На данном этапе test не используется для обучения или выбора модели.
Он нужен только для проверки совместимости feature pipeline и на финальном этапе — для формирования submission.

In [104]:
train = pd.read_csv(
    TRAIN_PATH,
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

test = pd.read_csv(
    TEST_PATH,
    parse_dates=[
        "cookie_created_at",
        "window_start_ts",
        "window_end_ts",
    ],
)

events = pd.read_csv(
    EVENTS_PATH,
    parse_dates=["event_ts"],
)

print("train:", train.shape)
print("test:", test.shape)
print("events:", events.shape)

train: (11091, 5)
test: (4909, 4)
events: (328905, 14)


In [105]:
display(train.head())
display(test.head())
display(events.head())

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


,cookie_id,cookie_created_at,window_start_ts,window_end_ts
0,ck_315fb710a0e371e7,2026-02-20 08:52:31,2026-04-20,2026-04-21
1,ck_a76ee3b3e3e522fd,2026-01-19 13:42:15,2026-04-20,2026-04-21
2,ck_94c9a4d382689e82,2026-04-19 06:28:37,2026-04-20,2026-04-21
3,ck_8eaf9509ad9462a0,2026-01-19 17:18:06,2026-04-20,2026-04-21
4,ck_9a88a5a989cb5bc6,2025-11-09 17:25:13,2026-04-20,2026-04-21


,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.000000,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.000000,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.000000,kvartiry_prodazha,kirov,private,NaN,NaN,675.000000,276.000000
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.000000,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.000000,NaN,NaN


## Проверка структуры данных и временных ограничений

Перед построением признаков проверяем:
- уникальность `cookie_id` в train/test;
- отсутствие пересечения train и test;
- отсутствие `target` в test;
- корректность временных окон;
- длительность каждого окна — ровно 24 часа.

In [106]:
assert train["cookie_id"].is_unique
assert test["cookie_id"].is_unique

assert set(train["cookie_id"]).isdisjoint(
    set(test["cookie_id"])
)

assert "target" in train.columns
assert "target" not in test.columns

assert (train["window_end_ts"] > train["window_start_ts"]).all()
assert (test["window_end_ts"] > test["window_start_ts"]).all()

assert (
    train["window_end_ts"] - train["window_start_ts"]
    == pd.Timedelta(days=1)
).all()

assert (
    test["window_end_ts"] - test["window_start_ts"]
    == pd.Timedelta(days=1)
).all()

assert train["target"].isin([0, 1]).all()

print("Input validation: OK")

Input validation: OK


## Проверка целевой переменной

В train целевая переменная содержит:

- `0` — traffic attributed to humans;
- `1` — traffic from known automated data-collection services.

Доля положительного класса используется только как диагностическая информация.

In [107]:
print("Target distribution:")
display(
    train["target"]
    .value_counts(normalize=True)
    .rename("share")
    .to_frame()
)

print("Positive rate:", train["target"].mean())

Target distribution:


,share
target,
0,0.918943
1,0.081057


Positive rate: 0.0810567126498963


## Формирование observation window

Для каждого cookie используются только события:

`window_start_ts <= event_ts < window_end_ts`

Это обязательное ограничение задачи.

События вне observation window не должны попадать в feature engineering.

In [108]:
def events_in_window(events, meta):
    meta_window = meta[
        [
            "cookie_id",
            "window_start_ts",
            "window_end_ts",
        ]
    ]

    ev = events.merge(
        meta_window,
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )

    mask = (
        (ev["event_ts"] >= ev["window_start_ts"])
        & (ev["event_ts"] < ev["window_end_ts"])
    )

    return ev.loc[mask].copy()

In [109]:
ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)

print("Train events in window:", len(ev_train))
print("Test events in window:", len(ev_test))

Train events in window: 198436
Test events in window: 89690


In [110]:
assert (
    ev_train["event_ts"] >= ev_train["window_start_ts"]
).all()

assert (
    ev_train["event_ts"] < ev_train["window_end_ts"]
).all()

assert (
    ev_test["event_ts"] >= ev_test["window_start_ts"]
).all()

assert (
    ev_test["event_ts"] < ev_test["window_end_ts"]
).all()

print("Event window validation: OK")

Event window validation: OK


In [111]:
ev_train = (
    ev_train
    .sort_values(
        ["cookie_id", "event_ts", "eid"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

ev_test = (
    ev_test
    .sort_values(
        ["cookie_id", "event_ts", "eid"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

## Нормализация platform

Значения `platform` встречаются в разных регистрах, поэтому для агрегатных признаков приводятся к единому виду.

In [112]:
for df in (ev_train, ev_test):
    df["platform_normalized"] = (
        df["platform"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

print(
    ev_train["platform_normalized"]
    .value_counts()
)

platform_normalized
web        82152
android    77991
desktop    27855
ios         7836
iphone      2602
Name: count, dtype: int64


# E03 Feature Set

Используется проверенный набор признаков из предыдущего эксперимента.

Он состоит из трёх групп:

1. metadata / cookie age;
2. activity и event distribution;
3. temporal behavior.

Основная идея:

**боты отличаются не только количеством событий, но и структурой и временным паттерном поведения.**

In [113]:
def add_metadata_features(meta):
    result = meta[
        [
            "cookie_id",
            "cookie_created_at",
            "window_start_ts",
            "window_end_ts",
        ]
    ].copy()

    result["cookie_age_seconds"] = (
        result["window_start_ts"]
        - result["cookie_created_at"]
    ).dt.total_seconds()

    result["cookie_age_hours"] = (
        result["cookie_age_seconds"] / 3600
    )

    return result

In [114]:
def build_activity_features(ev):
    g = ev.groupby("cookie_id", sort=False)

    features = pd.DataFrame({
        "n_events": g.size(),
        "n_unique_items": g["item_id"].nunique(),
        "n_unique_categories": g["item_category"].nunique(),
        "n_unique_locations": g["item_location"].nunique(),
        "n_unique_search_queries": g["search_query"].nunique(),
        "n_unique_event_types": g["event_name"].nunique(),
        "n_unique_platforms": g["platform_normalized"].nunique(),
    })

    return features.reset_index()

In [115]:
def build_event_count_features(ev):
    counts = pd.crosstab(
        ev["cookie_id"],
        ev["event_name"],
    )

    counts.columns = [
        f"event_count__{event_type}"
        for event_type in counts.columns
    ]

    return counts.reset_index()

In [116]:
def build_event_share_features(ev):
    counts = pd.crosstab(
        ev["cookie_id"],
        ev["event_name"],
    )

    shares = counts.div(
        counts.sum(axis=1),
        axis=0,
    )

    shares.columns = [
        f"event_share__{event_type}"
        for event_type in shares.columns
    ]

    return shares.reset_index()

In [117]:
def build_temporal_features(ev):
    df = ev[
        [
            "cookie_id",
            "event_ts",
            "window_start_ts",
            "window_end_ts",
        ]
    ].copy()

    df["event_offset_seconds"] = (
        df["event_ts"] - df["window_start_ts"]
    ).dt.total_seconds()

    df["inter_event_seconds"] = (
        df.groupby("cookie_id", sort=False)["event_ts"]
        .diff()
        .dt.total_seconds()
    )

    df["relative_hour"] = (
        df["event_offset_seconds"] // 3600
    ).astype(int)

    g = df.groupby("cookie_id", sort=False)

    features = g.agg(
        first_event_offset_seconds=(
            "event_offset_seconds",
            "min",
        ),
        last_event_offset_seconds=(
            "event_offset_seconds",
            "max",
        ),
        n_active_hours=(
            "relative_hour",
            "nunique",
        ),
        mean_inter_event_seconds=(
            "inter_event_seconds",
            "mean",
        ),
        median_inter_event_seconds=(
            "inter_event_seconds",
            "median",
        ),
        std_inter_event_seconds=(
            "inter_event_seconds",
            "std",
        ),
        min_inter_event_seconds=(
            "inter_event_seconds",
            "min",
        ),
        max_inter_event_seconds=(
            "inter_event_seconds",
            "max",
        ),
    ).reset_index()

    features["active_span_seconds"] = (
        features["last_event_offset_seconds"]
        - features["first_event_offset_seconds"]
    )

    # n_events рассчитывается здесь только для
    # производного temporal-признака.
    n_events = (
        g.size()
        .rename("n_events")
        .reset_index()
    )

    features = features.merge(
        n_events,
        on="cookie_id",
        how="left",
        validate="one_to_one",
    )

    features["events_per_active_hour"] = (
        features["n_events"]
        / (
            features["active_span_seconds"] / 3600
            + 1
        )
    )

    denominator = (
        features["std_inter_event_seconds"]
        + features["mean_inter_event_seconds"]
    )

    features["inter_event_burstiness"] = (
        (
            features["std_inter_event_seconds"]
            - features["mean_inter_event_seconds"]
        )
        / denominator.replace(0, np.nan)
    )

    interval_cols = [
        "mean_inter_event_seconds",
        "median_inter_event_seconds",
        "std_inter_event_seconds",
        "min_inter_event_seconds",
        "max_inter_event_seconds",
        "inter_event_burstiness",
    ]

    features[interval_cols] = (
        features[interval_cols]
        .fillna(0.0)
    )

    return features

## Сборка признаков

Все группы признаков объединяются по `cookie_id`.

Важно:

- каждая строка итоговой таблицы соответствует одному cookie;
- target присоединяется только для train;
- test target отсутствует;
- после объединения проверяется полное покрытие train/test.

In [118]:
def build_feature_table(meta, ev):
    metadata = add_metadata_features(meta)
    activity = build_activity_features(ev)
    event_counts = build_event_count_features(ev)
    event_shares = build_event_share_features(ev)
    temporal = build_temporal_features(ev)

    features = metadata.copy()

    for part in [
        activity,
        event_counts,
        event_shares,
        temporal,
    ]:
        features = features.merge(
            part,
            on="cookie_id",
            how="left",
            validate="one_to_one",
        )

    return features

In [119]:
X_train_raw = build_feature_table(train, ev_train)
X_test_raw = build_feature_table(test, ev_test)

print("X_train_raw:", X_train_raw.shape)
print("X_test_raw:", X_test_raw.shape)

X_train_raw: (11091, 43)
X_test_raw: (4909, 43)


In [120]:
train_ids = set(train["cookie_id"])
test_ids = set(test["cookie_id"])

feature_train_ids = set(X_train_raw["cookie_id"])
feature_test_ids = set(X_test_raw["cookie_id"])

print(
    "Missing train cookies:",
    len(train_ids - feature_train_ids),
)

print(
    "Extra train cookies:",
    len(feature_train_ids - train_ids),
)

print(
    "Missing test cookies:",
    len(test_ids - feature_test_ids),
)

print(
    "Extra test cookies:",
    len(feature_test_ids - test_ids),
)

Missing train cookies: 0
Extra train cookies: 0
Missing test cookies: 0
Extra test cookies: 0


# Feature Cleanup

На предыдущем E03 были обнаружены технические дубликаты:

- `n_events_x` / `n_events_y`;
- `cookie_age_seconds` / `cookie_age_hours`;
- некоторые признаки могли быть полностью идентичны друг другу.

Для финального сравнения моделей используем один канонический вариант feature table.

Правила очистки:

1. `cookie_id` не является признаком;
2. сохраняем `cookie_age_hours`;
3. сохраняем один `n_events`;
4. удаляем константные признаки;
5. удаляем точные дубликаты признаков;
6. не удаляем признаки только из-за корреляции — деревья и линейные модели могут использовать их по-разному.

In [121]:
def canonicalize_features(
    X_train_raw,
    X_test_raw,
):
    train_df = X_train_raw.copy()
    test_df = X_test_raw.copy()

    # One canonical representation of cookie age.
    if "cookie_age_seconds" in train_df.columns:
        train_df = train_df.drop(
            columns=["cookie_age_seconds"]
        )

    if "cookie_age_seconds" in test_df.columns:
        test_df = test_df.drop(
            columns=["cookie_age_seconds"]
        )

    # Keep one n_events column.
    n_events_candidates = [
        col
        for col in train_df.columns
        if col.startswith("n_events")
    ]

    if "n_events" not in train_df.columns:
        source = n_events_candidates[0]
        train_df["n_events"] = train_df[source]
        test_df["n_events"] = test_df[source]

    drop_n_events = [
        col
        for col in n_events_candidates
        if col != "n_events"
    ]

    train_df = train_df.drop(
        columns=drop_n_events,
        errors="ignore",
    )

    test_df = test_df.drop(
        columns=drop_n_events,
        errors="ignore",
    )

    # Remove identifiers from model matrix.
    feature_columns = [
        col
        for col in train_df.columns
        if col != "cookie_id"
    ]

    train_df = train_df[
        ["cookie_id"] + feature_columns
    ]

    test_df = test_df[
        ["cookie_id"] + feature_columns
    ]

    return train_df, test_df

In [122]:
X_train, X_test = canonicalize_features(
    X_train_raw,
    X_test_raw,
)

print("Before cleanup:", X_train_raw.shape)
print("After cleanup:", X_train.shape)

Before cleanup: (11091, 43)
After cleanup: (11091, 41)


In [123]:
feature_cols = [
    col
    for col in X_train.columns
    if col != "cookie_id"
]

missing_in_test = sorted(
    set(feature_cols) - set(X_test.columns)
)

extra_in_test = sorted(
    set(X_test.columns) - set(X_train.columns)
)

print("Missing in test:", missing_in_test)
print("Extra in test:", extra_in_test)

Missing in test: []
Extra in test: []


In [124]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (11091, 41)
Test shape: (4909, 41)


In [125]:
constant_features = [
    col
    for col in feature_cols
    if X_train[col].nunique(dropna=False) <= 1
]

print("Constant features:", len(constant_features))
print(constant_features)

Constant features: 0
[]


In [126]:
if constant_features:
    X_train = X_train.drop(
        columns=constant_features
    )
    X_test = X_test.drop(
        columns=constant_features
    )

feature_cols = [
    col
    for col in X_train.columns
    if col != "cookie_id"
]

print("Features after constant removal:", len(feature_cols))

Features after constant removal: 40


## Exact duplicate features

Если два признака имеют абсолютно одинаковые значения на всех train-объектах, один из них не несёт дополнительной информации.

Дубликаты определяются **только по train**, после чего те же признаки удаляются из test.

In [127]:
def find_duplicate_columns(df, columns):
    duplicate_groups = {}

    for i, col_a in enumerate(columns):
        for col_b in columns[i + 1:]:
            if df[col_a].equals(df[col_b]):
                duplicate_groups.setdefault(
                    col_a,
                    []
                ).append(col_b)

    return duplicate_groups

In [128]:
duplicate_groups = find_duplicate_columns(
    X_train,
    feature_cols,
)

print("Duplicate groups:", len(duplicate_groups))

for base_col, duplicates in duplicate_groups.items():
    print(base_col, "==", duplicates)

Duplicate groups: 0


In [129]:
duplicate_features = [
    duplicate
    for duplicates in duplicate_groups.values()
    for duplicate in duplicates
]

if duplicate_features:
    X_train = X_train.drop(
        columns=duplicate_features
    )
    X_test = X_test.drop(
        columns=duplicate_features
    )

feature_cols = [
    col
    for col in X_train.columns
    if col != "cookie_id"
    and pd.api.types.is_numeric_dtype(X_train[col])
]

print("Features after duplicate removal:", len(feature_cols))

Features after duplicate removal: 37


## Проверка качества feature matrix

Перед обучением проверяем:

- одинаковый набор признаков;
- отсутствие NaN;
- числовые типы;
- конечность значений;
- отсутствие `cookie_id` среди признаков.

In [130]:
X_train_model = X_train[feature_cols].copy()
X_test_model = X_test[feature_cols].copy()

print("Number of model features:", len(feature_cols))

print(
    "Train missing values:",
    X_train_model.isna().sum().sum(),
)

print(
    "Test missing values:",
    X_test_model.isna().sum().sum(),
)

print(
    "Train non-numeric:",
    X_train_model.select_dtypes(
        exclude=np.number
    ).columns.tolist(),
)

print(
    "Test non-numeric:",
    X_test_model.select_dtypes(
        exclude=np.number
    ).columns.tolist(),
)

print(
    "Train non-finite:",
    (~np.isfinite(X_train_model.to_numpy())).sum(),
)

print(
    "Test non-finite:",
    (~np.isfinite(X_test_model.to_numpy())).sum(),
)

Number of model features: 37
Train missing values: 0
Test missing values: 0
Train non-numeric: []
Test non-numeric: []
Train non-finite: 0
Test non-finite: 0


In [131]:
y = train["target"].astype(int)

print("X:", X_train_model.shape)
print("y:", y.shape)
print("Positive rate:", y.mean())

X: (11091, 37)
y: (11091,)
Positive rate: 0.0810567126498963


# Temporal Validation

Используется time-based validation.

Основной split:

- train: окна до `2026-04-17`;
- validation: окна с `2026-04-17`.

Это соответствует ранее использованному split и позволяет напрямую сравнить новые модели с E03.

Второй split будет использоваться только после основного сравнения моделей для проверки устойчивости результата.

In [132]:
VALID_START_A = pd.Timestamp("2026-04-17")

train_mask_a = (
    train["window_start_ts"] < VALID_START_A
)

valid_mask_a = (
    train["window_start_ts"] >= VALID_START_A
)

print("Train rows:", train_mask_a.sum())
print("Validation rows:", valid_mask_a.sum())

print(
    "Train positive rate:",
    y.loc[train_mask_a].mean(),
)

print(
    "Validation positive rate:",
    y.loc[valid_mask_a].mean(),
)

Train rows: 9140
Validation rows: 1951
Train positive rate: 0.08085339168490153
Validation positive rate: 0.08200922603792926


## Evaluation

Основная метрика — официальная `precision_at_recall`.

Важно:

мы не подбираем порог для submission.

Модель возвращает score / probability для каждого cookie, после чего официальный checker сам рассматривает необходимые thresholds.

Поэтому в этой части оценивается именно **качество ранжирования**.

In [133]:
def evaluate_scores(y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    return {
        "P@R>=0.70": precision_at_recall(
            y_true,
            scores,
            recall=0.70,
        ),
        "PR-AUC": average_precision_score(
            y_true,
            scores,
        ),
        "ROC-AUC": roc_auc_score(
            y_true,
            scores,
        ),
    }

# Model 1 — Logistic Regression

Логистическая регрессия используется как линейный benchmark.

Признаки имеют разные масштабы, поэтому используется `StandardScaler`.

`class_weight="balanced"` компенсирует дисбаланс классов.

Модель нужна прежде всего для проверки того, насколько задача решается линейной комбинацией признаков.

In [134]:
logreg_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=2000,
                random_state=SEED,
            ),
        ),
    ]
)

In [135]:
logreg_model.fit(
    X_train_model.loc[train_mask_a],
    y.loc[train_mask_a],
)

logreg_scores = logreg_model.predict_proba(
    X_train_model.loc[valid_mask_a]
)[:, 1]

logreg_metrics = evaluate_scores(
    y.loc[valid_mask_a],
    logreg_scores,
)

logreg_metrics

{'P@R>=0.70': 0.27053140096618356,
 'PR-AUC': 0.4437384910902691,
 'ROC-AUC': 0.8307754048017867}

# Model 2 — Random Forest

Random Forest уже использовался в quickstart и E03.

Здесь проверяется небольшая сетка по `min_samples_leaf`.

Большой grid search не используется, чтобы не подгонять модель под один validation split.

In [136]:
rf_configs = {
    "RF_leaf_1": 1,
    "RF_leaf_3": 3,
    "RF_leaf_5": 5,
    "RF_leaf_10": 10,
}

rf_models = {}
rf_results = []

In [137]:
for model_name, min_samples_leaf in rf_configs.items():
    model = RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1,
    )

    model.fit(
        X_train_model.loc[train_mask_a],
        y.loc[train_mask_a],
    )

    scores = model.predict_proba(
        X_train_model.loc[valid_mask_a]
    )[:, 1]

    metrics = evaluate_scores(
        y.loc[valid_mask_a],
        scores,
    )

    rf_models[model_name] = model

    rf_results.append({
        "model": model_name,
        **metrics,
    })

rf_results_df = (
    pd.DataFrame(rf_results)
    .sort_values(
        "P@R>=0.70",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(rf_results_df)

,model,P@R>=0.70,PR-AUC,ROC-AUC
0,RF_leaf_10,0.442688,0.600837,0.870652
1,RF_leaf_3,0.430769,0.616326,0.870247
2,RF_leaf_5,0.424812,0.610363,0.871200
3,RF_leaf_1,0.422642,0.619457,0.872997


# Model 3 — HistGradientBoosting

Gradient boosting рассматривается как следующий базовый вариант для табличных данных.

Используется одна умеренная конфигурация без большого hyperparameter search.

Цель — проверить сам класс модели, а не оптимизировать десятки параметров под validation.

In [138]:
hgb_model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=15,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=SEED,
)

In [139]:
hgb_model.fit(
    X_train_model.loc[train_mask_a],
    y.loc[train_mask_a],
)

hgb_scores = hgb_model.predict_proba(
    X_train_model.loc[valid_mask_a]
)[:, 1]

hgb_metrics = evaluate_scores(
    y.loc[valid_mask_a],
    hgb_scores,
)

hgb_metrics

{'P@R>=0.70': 0.44140625,
 'PR-AUC': 0.6758836316366486,
 'ROC-AUC': 0.8853852596314908}

# Model 4 — CatBoost

CatBoost проверяется как специализированная модель градиентного бустинга для табличных данных.

На этом этапе используются только числовые агрегированные признаки.

Конфигурация намеренно фиксированная:

- `iterations=500`;
- `depth=6`;
- `learning_rate=0.05`;
- фиксированный seed;
- без использования validation set для early stopping.

Это позволяет сравнивать модели на одном и том же validation split.

In [140]:
catboost_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    verbose=False,
    thread_count=-1,
    allow_writing_files=False,
)

In [141]:
catboost_model.fit(
    X_train_model.loc[train_mask_a],
    y.loc[train_mask_a],
)

catboost_scores = catboost_model.predict_proba(
    X_train_model.loc[valid_mask_a]
)[:, 1]

catboost_metrics = evaluate_scores(
    y.loc[valid_mask_a],
    catboost_scores,
)

catboost_metrics

{'P@R>=0.70': 0.43410852713178294,
 'PR-AUC': 0.6742930806776747,
 'ROC-AUC': 0.88500139586823}

# Model Comparison

Основная сортировка выполняется по `P@R>=0.70`.

PR-AUC и ROC-AUC используются как дополнительные показатели качества ранжирования.

In [142]:
model_results = [
    {
        "model": "LogisticRegression",
        **logreg_metrics,
    },
    {
        "model": "HistGradientBoosting",
        **hgb_metrics,
    },
    {
        "model": "CatBoost",
        **catboost_metrics,
    },
]

model_results.extend(rf_results)

model_results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        "P@R>=0.70",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(model_results_df)

,model,P@R>=0.70,PR-AUC,ROC-AUC
0,RF_leaf_10,0.442688,0.600837,0.870652
1,HistGradientBoosting,0.441406,0.675884,0.885385
2,CatBoost,0.434109,0.674293,0.885001
3,RF_leaf_3,0.430769,0.616326,0.870247
4,RF_leaf_5,0.424812,0.610363,0.871200
5,RF_leaf_1,0.422642,0.619457,0.872997
6,LogisticRegression,0.270531,0.443738,0.830775


## Comparison with E03

Исторический результат E03:

- P@R>=0.70: `0.332264`
- PR-AUC: `0.606980`
- ROC-AUC: `0.850155`

Он используется как benchmark для нового этапа.

Если новая модель не превосходит E03 по основной метрике, это не означает, что feature set необходимо немедленно расширять: следующим шагом является проверка устойчивости существующего результата.

In [143]:
E03_REFERENCE = {
    "model": "E03_reference",
    "P@R>=0.70": 0.33226427196921104,
    "PR-AUC": 0.6069801746377597,
    "ROC-AUC": 0.8501553314018452,
}

comparison_with_e03 = pd.concat(
    [
        pd.DataFrame([E03_REFERENCE]),
        model_results_df,
    ],
    ignore_index=True,
)

display(comparison_with_e03)

,model,P@R>=0.70,PR-AUC,ROC-AUC
0,E03_reference,0.332264,0.606980,0.850155
1,RF_leaf_10,0.442688,0.600837,0.870652
2,HistGradientBoosting,0.441406,0.675884,0.885385
3,CatBoost,0.434109,0.674293,0.885001
4,RF_leaf_3,0.430769,0.616326,0.870247
5,RF_leaf_5,0.424812,0.610363,0.871200
6,RF_leaf_1,0.422642,0.619457,0.872997
7,LogisticRegression,0.270531,0.443738,0.830775


# Robustness Check

После основного сравнения не следует проверять все модели на множестве дополнительных split'ов.

Берём две лучшие модели по основной метрике и проверяем их на втором временном split.

Это снижает риск выбора модели, которая случайно хорошо работает только на последних трёх днях первого validation периода.

In [144]:
def create_model(model_name):
    if model_name == "LogisticRegression":
        return Pipeline(
            steps=[
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "model",
                    LogisticRegression(
                        C=1.0,
                        class_weight="balanced",
                        max_iter=2000,
                        random_state=SEED,
                    ),
                ),
            ]
        )

    if model_name == "RF_leaf_10":
        return RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        )

    if model_name == "HistGradientBoosting":
        return HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=15,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=SEED,
        )

    if model_name == "CatBoost":
        return CatBoostClassifier(
            iterations=500,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=SEED,
            verbose=False,
            thread_count=-1,
            allow_writing_files=False,
        )

    raise ValueError(
        f"Unknown model: {model_name}"
    )

## Walk-forward validation

Поскольку test-период следует непосредственно после train-периода, основная проверка устойчивости выполняется по времени.

Для каждой даты:
- обучение использует только более ранние наблюдения;
- валидация использует только текущий день;
- будущие данные не используются.

Сравниваются три кандидата:
- Random Forest;
- HistGradientBoosting;
- CatBoost.

Основная метрика — Precision при Recall >= 0.70.
PR-AUC и ROC-AUC используются как дополнительные диагностические метрики.

In [145]:
WALK_FORWARD_DATES = [
    pd.Timestamp("2026-04-17"),
    pd.Timestamp("2026-04-18"),
    pd.Timestamp("2026-04-19"),
]

CANDIDATE_MODELS = [
    "RF_leaf_10",
    "HistGradientBoosting",
    "CatBoost",
]

In [146]:
walk_forward_results = []
oof_predictions = []

for validation_date in WALK_FORWARD_DATES:
    train_mask = (
        train["window_start_ts"] < validation_date
    )

    valid_mask = (
        train["window_start_ts"] == validation_date
    )

    assert train_mask.sum() > 0
    assert valid_mask.sum() > 0

    for model_name in CANDIDATE_MODELS:
        model = create_model(model_name)

        model.fit(
            X_train_model.loc[train_mask],
            y.loc[train_mask],
        )

        valid_scores = model.predict_proba(
            X_train_model.loc[valid_mask]
        )[:, 1]

        metrics = evaluate_scores(
            y.loc[valid_mask],
            valid_scores,
        )

        walk_forward_results.append(
            {
                "validation_date": validation_date,
                "model": model_name,
                **metrics,
                "n_train": int(train_mask.sum()),
                "n_valid": int(valid_mask.sum()),
                "positive_rate_valid": float(
                    y.loc[valid_mask].mean()
                ),
            }
        )

        oof_predictions.append(
            pd.DataFrame(
                {
                    "cookie_id": train.loc[
                        valid_mask, "cookie_id"
                    ].values,
                    "validation_date": validation_date,
                    "model": model_name,
                    "target": y.loc[
                        valid_mask
                    ].values,
                    "score": valid_scores,
                }
            )
        )

walk_forward_df = (
    pd.DataFrame(walk_forward_results)
    .sort_values(
        ["model", "validation_date"]
    )
    .reset_index(drop=True)
)

display(walk_forward_df)

,validation_date,model,P@R>=0.70,PR-AUC,ROC-AUC,n_train,n_valid,positive_rate_valid
0,2026-04-17,CatBoost,0.427273,0.664358,0.849679,9140,713,0.092567
1,2026-04-18,CatBoost,0.468354,0.654977,0.884164,9853,606,0.085809
2,2026-04-19,CatBoost,0.666667,0.710813,0.950767,10459,632,0.066456
3,2026-04-17,HistGradientBoosting,0.447619,0.673058,0.864948,9140,713,0.092567
4,2026-04-18,HistGradientBoosting,0.411111,0.671143,0.875417,9853,606,0.085809
5,2026-04-19,HistGradientBoosting,0.612245,0.712695,0.945157,10459,632,0.066456
6,2026-04-17,RF_leaf_10,0.370079,0.618132,0.851014,9140,713,0.092567
7,2026-04-18,RF_leaf_10,0.352381,0.556660,0.846293,9853,606,0.085809
8,2026-04-19,RF_leaf_10,0.508475,0.628242,0.936642,10459,632,0.066456


In [147]:
walk_forward_summary = (
    walk_forward_df
    .groupby("model")
    .agg(
        P_at_R_mean=(
            "P@R>=0.70",
            "mean",
        ),
        P_at_R_std=(
            "P@R>=0.70",
            "std",
        ),
        PR_AUC_mean=(
            "PR-AUC",
            "mean",
        ),
        PR_AUC_std=(
            "PR-AUC",
            "std",
        ),
        ROC_AUC_mean=(
            "ROC-AUC",
            "mean",
        ),
        ROC_AUC_std=(
            "ROC-AUC",
            "std",
        ),
    )
    .reset_index()
)

display(
    walk_forward_summary
    .sort_values(
        "P_at_R_mean",
        ascending=False,
    )
)

,model,P_at_R_mean,P_at_R_std,PR_AUC_mean,PR_AUC_std,ROC_AUC_mean,ROC_AUC_std
0,CatBoost,0.520765,0.128014,0.676716,0.029899,0.894870,0.051387
1,HistGradientBoosting,0.490325,0.107152,0.685632,0.023457,0.895174,0.043602
2,RF_leaf_10,0.410311,0.085471,0.601011,0.038740,0.877983,0.050855


In [148]:
oof_predictions_df = pd.concat(
    oof_predictions,
    ignore_index=True,
)

pooled_results = []

for model_name in CANDIDATE_MODELS:
    model_oof = oof_predictions_df[
        oof_predictions_df["model"] == model_name
    ]

    metrics = evaluate_scores(
        model_oof["target"],
        model_oof["score"],
    )

    pooled_results.append(
        {
            "model": model_name,
            **metrics,
            "n_oof": len(model_oof),
            "positive_rate_oof": model_oof[
                "target"
            ].mean(),
        }
    )

pooled_results_df = (
    pd.DataFrame(pooled_results)
    .sort_values(
        "P@R>=0.70",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(pooled_results_df)

,model,P@R>=0.70,PR-AUC,ROC-AUC,n_oof,positive_rate_oof
0,CatBoost,0.457490,0.669516,0.885560,1951,0.082009
1,HistGradientBoosting,0.442688,0.680734,0.886125,1951,0.082009
2,RF_leaf_10,0.430769,0.595518,0.872138,1951,0.082009


In [149]:
final_model_comparison = (
    walk_forward_summary
    .merge(
        pooled_results_df[
            [
                "model",
                "P@R>=0.70",
                "PR-AUC",
                "ROC-AUC",
            ]
        ].rename(
            columns={
                "P@R>=0.70": "P@R>=0.70_pooled",
                "PR-AUC": "PR-AUC_pooled",
                "ROC-AUC": "ROC-AUC_pooled",
            }
        ),
        on="model",
        how="left",
    )
)

display(
    final_model_comparison.sort_values(
        [
            "P@R>=0.70_pooled",
            "P_at_R_mean",
        ],
        ascending=False,
    )
)

,model,P_at_R_mean,P_at_R_std,PR_AUC_mean,PR_AUC_std,ROC_AUC_mean,ROC_AUC_std,P@R>=0.70_pooled,PR-AUC_pooled,ROC-AUC_pooled
0,CatBoost,0.520765,0.128014,0.676716,0.029899,0.894870,0.051387,0.457490,0.669516,0.885560
1,HistGradientBoosting,0.490325,0.107152,0.685632,0.023457,0.895174,0.043602,0.442688,0.680734,0.886125
2,RF_leaf_10,0.410311,0.085471,0.601011,0.038740,0.877983,0.050855,0.430769,0.595518,0.872138


In [150]:
model_selection_table = (
    final_model_comparison
    .sort_values(
        [
            "P@R>=0.70_pooled",
            "P_at_R_mean",
            "PR_AUC_mean",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(model_selection_table)

best_model_name = model_selection_table.loc[
    0,
    "model",
]

print("Selected final model:", best_model_name)

,model,P_at_R_mean,P_at_R_std,PR_AUC_mean,PR_AUC_std,ROC_AUC_mean,ROC_AUC_std,P@R>=0.70_pooled,PR-AUC_pooled,ROC-AUC_pooled
0,CatBoost,0.520765,0.128014,0.676716,0.029899,0.894870,0.051387,0.457490,0.669516,0.885560
1,HistGradientBoosting,0.490325,0.107152,0.685632,0.023457,0.895174,0.043602,0.442688,0.680734,0.886125
2,RF_leaf_10,0.410311,0.085471,0.601011,0.038740,0.877983,0.050855,0.430769,0.595518,0.872138


Selected final model: CatBoost


In [151]:
selected_row = model_selection_table.iloc[0]

print(
    "Final model selection:"
)
print(
    "Model:",
    selected_row["model"],
)
print(
    "P@R pooled:",
    selected_row["P@R>=0.70_pooled"],
)
print(
    "Mean daily P@R:",
    selected_row["P_at_R_mean"],
)
print(
    "Std daily P@R:",
    selected_row["P_at_R_std"],
)
print(
    "PR-AUC pooled:",
    selected_row["PR-AUC_pooled"],
)
print(
    "ROC-AUC pooled:",
    selected_row["ROC-AUC_pooled"],
)

Final model selection:
Model: CatBoost
P@R pooled: 0.4574898785425101
Mean daily P@R: 0.5207646081063803
Std daily P@R: 0.12801361100148786
PR-AUC pooled: 0.6695163803580241
ROC-AUC pooled: 0.8855597431602455


# Final Model Fit

После выбора модели она обучается на всём доступном train.

Validation split больше не используется для обучения отдельной модели: все размеченные train-данные доступны финальному обучению.

Test используется только для получения итоговых score.

In [152]:
final_model = create_model(
    best_model_name
)

assert list(X_train_model.columns) == feature_cols
assert list(X_test_model.columns) == feature_cols

assert X_train_model.shape[0] == len(train)
assert X_test_model.shape[0] == len(test)

assert X_train_model.isna().sum().sum() == 0
assert X_test_model.isna().sum().sum() == 0

assert np.isfinite(
    X_train_model.to_numpy()
).all()

assert np.isfinite(
    X_test_model.to_numpy()
).all()

print("Final feature matrix validation: OK")

final_model.fit(
    X_train_model,
    y,
)

print(
    "Final model fitted:",
    best_model_name,
)

Final feature matrix validation: OK
Final model fitted: CatBoost


In [153]:
test_scores = final_model.predict_proba(
    X_test_model
)[:, 1]

assert len(test_scores) == len(test)

assert np.isfinite(
    test_scores
).all()

assert (
    (test_scores >= 0)
    & (test_scores <= 1)
).all()

print("Test score validation: OK")
print("Number of scores:", len(test_scores))
print("Min:", test_scores.min())
print("Max:", test_scores.max())
print("Mean:", test_scores.mean())

Test score validation: OK
Number of scores: 4909
Min: 0.0007512469235884593
Max: 0.9997391966254203
Mean: 0.08113387739511504


## Score

`score` интерпретируется как степень принадлежности к положительному классу:

- больше score → выше приоритет как потенциального automated traffic;
- меньше score → ниже приоритет.

Для submission не выбирается фиксированный threshold.

Передаётся полный ranking score для всех test cookies.

In [154]:
submission = pd.DataFrame({
    "cookie_id": test["cookie_id"],
    "score": test_scores,
})

submission.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.003116
1,ck_a76ee3b3e3e522fd,0.035317
2,ck_94c9a4d382689e82,0.021563
3,ck_8eaf9509ad9462a0,0.011677
4,ck_9a88a5a989cb5bc6,0.008684


# Submission Validation

Проверяем обязательные требования:

- ровно две колонки;
- `cookie_id`;
- `score`;
- одна строка на каждый test cookie;
- отсутствие дубликатов;
- отсутствие NaN;
- все test cookies присутствуют;
- score находится в диапазоне `[0, 1]`.

In [155]:
assert list(submission.columns) == [
    "cookie_id",
    "score",
]

assert len(submission) == len(test)

assert submission["cookie_id"].is_unique
assert submission["cookie_id"].notna().all()

assert submission["score"].notna().all()

assert np.isfinite(
    submission["score"].to_numpy()
).all()

assert submission["score"].between(
    0,
    1,
).all()

assert submission["cookie_id"].equals(
    test["cookie_id"]
)

print("Submission validation: OK")

Submission validation: OK


In [156]:
SUBMISSION_PATH = ROOT_DIR / "submission.csv"

submission.to_csv(
    SUBMISSION_PATH,
    index=False,
)

print("Saved:", SUBMISSION_PATH)
print("Shape:", submission.shape)

Saved: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge\submission.csv
Shape: (4909, 2)


## Saving experiment results

Результаты экспериментов сохраняются отдельно, чтобы финальный выбор модели был воспроизводим и документирован.

In [157]:
RESULTS_DIR = ROOT_DIR / "results"
RESULTS_DIR.mkdir(
    exist_ok=True
)

model_results_df.to_csv(
    RESULTS_DIR / "model_results_split_A.csv",
    index=False,
)

walk_forward_df.to_csv(
    RESULTS_DIR / "walk_forward_results.csv",
    index=False,
)

walk_forward_summary.to_csv(
    RESULTS_DIR / "walk_forward_summary.csv",
    index=False,
)

pooled_results_df.to_csv(
    RESULTS_DIR / "pooled_oof_results.csv",
    index=False,
)

final_model_comparison.to_csv(
    RESULTS_DIR / "final_model_comparison.csv",
    index=False,
)

print(
    "Saved experiment results to:",
    RESULTS_DIR,
)

Saved experiment results to: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge\results


In [158]:
FEATURES_PATH = RESULTS_DIR / "final_feature_list.txt"

FEATURES_PATH.write_text(
    "\n".join(feature_cols),
    encoding="utf-8",
)

print("Saved feature list:", FEATURES_PATH)
print("Number of final features:", len(feature_cols))

Saved feature list: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge\results\final_feature_list.txt
Number of final features: 37


In [159]:
FINAL_CONFIG_PATH = (
    RESULTS_DIR / "final_model_config.txt"
)

FINAL_CONFIG_PATH.write_text(
    "\n".join(
        [
            f"model={best_model_name}",
            f"seed={SEED}",
            f"n_features={len(feature_cols)}",
            f"n_train={len(train)}",
            f"n_test={len(test)}",
            "validation=walk_forward",
            "primary_metric=precision_at_recall_0.70",
            "score_source=predict_proba[:,1]",
        ]
    ),
    encoding="utf-8",
)

print(
    "Saved final model config:",
    FINAL_CONFIG_PATH,
)

Saved final model config: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge\results\final_model_config.txt


In [160]:
import importlib.metadata as metadata

required_packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "catboost",
]

requirements = []

for package in required_packages:
    requirements.append(
        f"{package}=={metadata.version(package)}"
    )

REQUIREMENTS_PATH = (
    ROOT_DIR / "requirements.txt"
)

REQUIREMENTS_PATH.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8",
)

print(
    "Saved requirements:",
    REQUIREMENTS_PATH,
)

print(
    REQUIREMENTS_PATH.read_text(
        encoding="utf-8"
    )
)

Saved requirements: \\wsl.localhost\Ubuntu-24.04\home\ilush\Avito\test_assignment\bot-detection-challenge\requirements.txt
numpy==2.1.1
pandas==2.2.3
scikit-learn==1.7.2
catboost==1.2.10



In [161]:
print("=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Features:", len(feature_cols))
print("Final model:", best_model_name)

print()
print("Primary metric:")
print("Precision @ Recall >= 0.70")

print()
print("Submission:")
print("Rows:", len(submission))
print("Columns:", list(submission.columns))
print(
    "Scores in [0, 1]:",
    submission["score"].between(0, 1).all(),
)

print("=" * 70)

FINAL VALIDATION
Train rows: 11091
Test rows: 4909
Features: 37
Final model: CatBoost

Primary metric:
Precision @ Recall >= 0.70

Submission:
Rows: 4909
Columns: ['cookie_id', 'score']
Scores in [0, 1]: True
